# ARIA Fine-Tuning on Google Colab (Free T4 GPU)
Runtime > Change runtime type > T4 GPU
Then Run All (Ctrl+F9)

In [ ]:
!pip install unsloth trl peft accelerate datasets -q

In [ ]:
import base64, json
# Dataset: 30 examples (base64-encoded to avoid control-char issues)
_b64 = '...'
dataset_jsonl = base64.b64decode(_b64).decode('utf-8')
with open('aria_dataset.jsonl', 'w', encoding='utf-8') as f:
    f.write(dataset_jsonl)
print(f'Loaded 30 training examples')

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=1024,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing=True,
)

examples = [json.loads(l) for l in open('aria_dataset.jsonl') if l.strip()]
ds = Dataset.from_list(examples)
ds = ds.map(lambda ex: {'text': tokenizer.apply_chat_template(
    ex['messages'], tokenize=False, add_generation_prompt=False)},
    remove_columns=['messages'])

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    dataset_text_field='text', max_seq_length=1024,
    args=TrainingArguments(output_dir='aria-lora', num_train_epochs=2,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        learning_rate=2e-4, fp16=True, logging_steps=5,
        save_strategy='epoch', report_to='none'),
)
trainer.train()
model.save_pretrained('aria-lora')
tokenizer.save_pretrained('aria-lora')
print('Training complete!')

In [ ]:
# Save LoRA adapter as GGUF and download
model.save_pretrained_gguf('aria-gguf', tokenizer, quantization_method='q4_k_m')
from google.colab import files
import os
for f in os.listdir('aria-gguf'):
    if f.endswith('.gguf'):
        files.download(f'aria-gguf/{f}')
        print(f'Download started: {f}')

## After download:
1. Move the .gguf file to `C:/Users/chand/ai-remo/data/models/gguf/`
2. Run: `python training/register_gguf.py --file aria-custom-q4.gguf`
3. Restart server.py — ARIA will use your fine-tuned model